# Fig. 4 and Tables S1–S2 — isolate mutation summaries

Mutation calling and driver selection are performed by `raw_processing/process_isolates/step2_identify_isolate_mutations.py`. This notebook only inspects its processed outputs; figure plotting is in `4a_isolate_seq_plotting.ipynb`.

In [1]:
import sys; sys.path.append('..') ## proejct root
from methods.config import *
import methods.genomics_util as genomics

import pickle
import numpy as np

with open(f'{pickled_dir}/barcode_clone_map.pkl', 'rb') as f:
    barcode_clone_map = pickle.load(f)
with open(f'{pickled_dir}/clone_barcode_map.pkl', 'rb') as f:
    clone_barcode_map = pickle.load(f)
with open(f'{pickled_dir}/all_mutations_in_isolates.pkl', 'rb') as f:
    all_mutations_in_isolates = pickle.load(f)
with open(f'{pickled_dir}/all_vivo_drivers.pkl', 'rb') as f:
    all_vivo_drivers = pickle.load(f)
with open(f'{pickled_dir}/SV_pvals.pkl', 'rb') as f:
    SV_pvals = pickle.load(f)

## Structural-variant uniformity across diets

This downstream analysis tests whether each structural variant is enriched in MD (mice >10) or SD (mice ≤10), at both the isolate and barcode levels. It does not affect candidate-driver selection.

In [ ]:

def adjust_bh(pvalues):
    adjusted = np.empty(len(pvalues), dtype=float)
    order = np.argsort(pvalues)
    previous = 1.0
    for rank_index in range(len(order) - 1, -1, -1):
        index = order[rank_index]
        rank = rank_index + 1
        previous = min(previous, pvalues[index] * len(order) / rank, 1.0)
        adjusted[index] = previous
    return adjusted

clone_diet = {clone: ('MD' if clone[0] > 10 else 'SD') for clone in clone_barcode_map}
barcode_diet = {}
for barcode, clone_data in barcode_clone_map.items():
    diets = {clone_diet[clone] for clone in clone_data['all']}
    if len(diets) != 1:
        raise ValueError(f'Barcode spans diets: {barcode} {diets}')
    barcode_diet[barcode] = diets.pop()

clone_totals = {diet: sum(value == diet for value in clone_diet.values()) for diet in ('SD', 'MD')}
barcode_totals = {diet: sum(value == diet for value in barcode_diet.values()) for diet in ('SD', 'MD')}

In [ ]:
diet_rows = []
for structural_variant, (variant_clones, variant_barcodes, _, _) in SV_pvals.items():
    if len(variant_barcodes) < 2:
        continue
    clone_present = {diet: sum(clone_diet[clone] == diet for clone in variant_clones) for diet in ('SD', 'MD')}
    barcode_present = {diet: sum(barcode_diet[barcode] == diet for barcode in variant_barcodes) for diet in ('SD', 'MD')}
    clone_table = [[clone_totals[diet] - clone_present[diet], clone_present[diet]] for diet in ('SD', 'MD')]
    barcode_table = [[barcode_totals[diet] - barcode_present[diet], barcode_present[diet]] for diet in ('SD', 'MD')]
    clone_pvalue = scipy.stats.fisher_exact(clone_table).pvalue
    barcode_pvalue = scipy.stats.fisher_exact(barcode_table).pvalue
    diet_rows.append([str(structural_variant), *np.ravel(clone_table), clone_pvalue,
                      *np.ravel(barcode_table), barcode_pvalue])

if diet_rows:
    clone_qvalues = adjust_bh(np.array([row[5] for row in diet_rows]))
    barcode_qvalues = adjust_bh(np.array([row[10] for row in diet_rows]))
    for row, clone_qvalue, barcode_qvalue in zip(diet_rows, clone_qvalues, barcode_qvalues):
        row.insert(6, clone_qvalue)
        row.append(barcode_qvalue)

diet_table = f'{tables_dir}/isolate_SV_diet_uniformity.tsv'
header = ['SV', 'SD_clones_absent', 'SD_clones_present', 'MD_clones_absent', 'MD_clones_present',
          'clone_Fisher_p', 'clone_BH_q', 'SD_barcodes_absent', 'SD_barcodes_present',
          'MD_barcodes_absent', 'MD_barcodes_present', 'barcode_Fisher_p', 'barcode_BH_q']
with open(diet_table, 'w') as f:
    f.write('\t'.join(header) + '\n')
    for row in diet_rows:
        f.write('\t'.join(map(str, row)) + '\n')
print(f'Wrote {len(diet_rows)} tests to {diet_table}')

## Processing summary

In [ ]:
barcode_driver_map = {barcode: [] for barcode in barcode_clone_map}
for mutation, barcode_map in all_vivo_drivers.items():
    for barcode in barcode_map:
        barcode_driver_map[barcode].append(mutation)

categories = {'simple only': 0, 'SV only': 0, 'inversion only': 0,
              'simple + SV': 0, 'inversion + other': 0, 'none': 0}
for mutations in barcode_driver_map.values():
    simple = sum(mutation[-1] not in {'unpaired_JC', 'insertion', 'SV', 'IR-inversion'} for mutation in mutations)
    structural = sum(mutation[-1] in {'unpaired_JC', 'insertion', 'SV'} for mutation in mutations)
    inversions = sum(mutation[-1] == 'IR-inversion' for mutation in mutations)
    if inversions and not simple and not structural:
        categories['inversion only'] += 1
    elif simple and not structural and not inversions:
        categories['simple only'] += 1
    elif structural and not simple and not inversions:
        categories['SV only'] += 1
    elif simple and structural and not inversions:
        categories['simple + SV'] += 1
    elif inversions:
        categories['inversion + other'] += 1
    else:
        categories['none'] += 1

print(f'Number of barcodes: {len(barcode_clone_map)}')
print(f'Number of clones: {len(clone_barcode_map)}')
print(f'Filtered mutations: {len(all_mutations_in_isolates)}')
print(f'Candidate drivers: {len(all_vivo_drivers)}')
for label, count in categories.items():
    print(f'{label}: {count}')

## Mutations in highlighted Fig. 2 lineages

In [ ]:
figure_examples = {
    'AGACGACAATATCCACTTTC': 'Fig. 2E',
    'TCCGTTAACCTTCATAGTTG': 'Fig. 2F',
    'CGGTTGCGGTATTACAAGTC': 'Fig. 2G',
    'ATTAACCGTTGACCGCTGCT': 'Fig. 2H',
    'TGTAGTAGGCATAATAACCC': 'Fig. 2I',
    'AACTACTCTTTGCATTTCCG': 'Fig. 2J',
}
for mutation, barcode_map in all_vivo_drivers.items():
    labels = [figure_examples[barcode] for barcode in barcode_map if barcode in figure_examples]
    if labels and mutation[-1] != 'True':
        print(mutation, labels)

## Optional annotation exports

These cells are downstream inspection utilities and are not required to regenerate the processed mutation artifacts.

In [ ]:
driver_genes = set()
for mutation in all_vivo_drivers:
    if mutation[-1] in {'unpaired_JC', 'insertion', 'SV', 'IR-inversion'}:
        continue
    driver_genes.update(mutation[2].replace('/', '-').split('-'))
for gene in sorted(driver_genes):
    pul = genomics.gene_PUL_map.get(gene, ('',))[0]
    print(gene.replace('BT', 'BT_'), pul, genomics.gene_description.get(gene, ''))